# MultiMNIST: focused late-learning-rate refinement (A100)

Run cells **1–8 in order**. After a disconnect, rerun cells 1–8; completed runs are verified and skipped, and interrupted training resumes from the last completed epoch.

This comparison-only notebook reuses all 18 completed baseline runs and tunes **only Entropic LMO-MGDA**. Ablations remain in `MultiMNIST_Ablations_A100.ipynb`.

The checkpoint audit found large late-training updates. A controlled two-epoch **validation-only diagnostic** improved from 87.83% at LR 0.01 to 90.68% at LR 0.003 and 90.72% at LR 0.001. These are extra-epoch diagnostic numbers, **not final test results or table entries**.

The actual comparison keeps **100 epochs, batch size 256, the same ViT, the same generated dataset and seeds 42/43/44**. Four schedules keep LR constant for 60 or 80 epochs, then decay to 10% or 30% of the initial LR. Every candidate is evaluated on all three validation seeds. The saved incumbent is retained unless validation improves by more than 0.1 percentage point. The oracle, task-weight rule and momentum are unchanged.

Cell 6: 12 validation runs. Cell 7: three fresh full-training runs only if the validation-selected recipe changes. Budget roughly 45–60 minutes based on the previous A100 epoch times; actual times depend on the runtime. Logs stream live once per epoch. No baseline training is launched.

The previous experiment stays under `ours_retune_v3`; new outputs go under `ours_refine_v4`. Do not rerun the old broad search. No test-superiority guarantee is made.


In [ ]:
#@title 1. Fixed experiment settings
REPO_URL = "https://github.com/alirezamirrokni/LMO-MOO.git"
REPO_COMMIT = "dec6f3b808281276d12d08a1de312d54a1aa53cc"
EXPERIMENT_NAME = "multimnist_a100_v2" #@param {type:"string"}
SEARCH_NAME = "ours_refine_v4" #@param {type:"string"}
PREVIOUS_SEARCH_NAME = "ours_retune_v3"
DATA_CACHE_EXPERIMENT = "multimnist_a100_v1"
SEEDS = [42, 43, 44]
EPOCHS = 100
BATCH_SIZE = 256


In [ ]:
#@title 2. Check A100, mount Drive and enable live console logs
import os, sys, json, subprocess, shutil, time, hashlib, zipfile, csv
from pathlib import Path
import torch
from google.colab import drive

assert torch.cuda.is_available(), "Select Runtime > Change runtime type > A100 GPU."
GPU_NAME = torch.cuda.get_device_name(0)
assert "A100" in GPU_NAME, f"Current GPU: {GPU_NAME}. Select A100 and reconnect."
print("GPU:", GPU_NAME, "| PyTorch:", torch.__version__)
drive.mount("/content/drive")
assert EXPERIMENT_NAME and Path(EXPERIMENT_NAME).name == EXPERIMENT_NAME and EXPERIMENT_NAME not in {".", ".."}
DRIVE_ROOT = Path("/content/drive/MyDrive/LMO-MOO")
BASE_ROOT = DRIVE_ROOT / EXPERIMENT_NAME
assert SEARCH_NAME and Path(SEARCH_NAME).name == SEARCH_NAME and SEARCH_NAME not in {".", ".."}
RUN_ROOT = BASE_ROOT / SEARCH_NAME
RUN_ROOT.mkdir(parents=True, exist_ok=True)
BASELINE_ROOT = BASE_ROOT / "results" / "multimnist"
TUNING_ROOT = RUN_ROOT / "tuning"
SELECTION_FILE = TUNING_ROOT / "selection.json"
INCUMBENT_SELECTION = BASE_ROOT / "tuning" / "selection.json"
OUTPUT_ROOT = RUN_ROOT / "results" / "multimnist"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
REPO = Path("/content/LMO-MOO-refine-v4")
DATA = Path("/content/LMO-MOO-data/multimnist")
print("Persistent outputs:", OUTPUT_ROOT)


import codecs, signal, shlex, uuid
LOG_DIR = RUN_ROOT / "logs"
LOG_DIR.mkdir(exist_ok=True)

def run_live(cmd, *, cwd=None, label="process"):
    """Forward stdout/stderr chunks immediately, preserving carriage returns.

    Raw console logs are also saved to Drive. An interrupted cell stops the
    entire subprocess group, including children launched by the suite.
    """
    cmd = list(map(str, cmd))
    print("$ " + shlex.join(cmd), flush=True)
    safe_label = "".join(ch if ch.isalnum() or ch in "-_" else "_" for ch in label)
    log_path = LOG_DIR / (time.strftime("%Y%m%d_%H%M%S") + "_" + safe_label + "_" + uuid.uuid4().hex[:6] + ".log")
    env = os.environ.copy()
    env.update(PYTHONUNBUFFERED="1", PYTHONIOENCODING="utf-8")
    print("Console log:", log_path, flush=True)
    with log_path.open("wb", buffering=0) as log:
        process = subprocess.Popen(cmd, cwd=cwd, env=env, stdout=subprocess.PIPE,
                                   stderr=subprocess.STDOUT, bufsize=0,
                                   start_new_session=True)
        decoder = codecs.getincrementaldecoder("utf-8")(errors="replace")
        try:
            while True:
                chunk = os.read(process.stdout.fileno(), 8192)
                if not chunk:
                    break
                # Display first so a slow Drive write cannot hide current output.
                sys.stdout.write(decoder.decode(chunk))
                sys.stdout.flush()
                log.write(chunk)
            sys.stdout.write(decoder.decode(b"", final=True))
            sys.stdout.flush()
            returncode = process.wait()
        except BaseException:
            if process.poll() is None:
                try:
                    os.killpg(process.pid, signal.SIGTERM)
                except ProcessLookupError:
                    pass
                try:
                    process.wait(timeout=5)
                except (subprocess.TimeoutExpired, KeyboardInterrupt):
                    try:
                        os.killpg(process.pid, signal.SIGKILL)
                    except ProcessLookupError:
                        pass
                    process.wait()
            raise
        finally:
            process.stdout.close()
    print(f"\n[{label}] Exit code: {returncode}", flush=True)
    if returncode:
        raise subprocess.CalledProcessError(returncode, cmd)
    return log_path
PREVIOUS_ROOT = BASE_ROOT / PREVIOUS_SEARCH_NAME
PREVIOUS_SEARCH = PREVIOUS_ROOT / "tuning"
PREVIOUS_SELECTION = PREVIOUS_SEARCH / "selection.json"
assert PREVIOUS_SELECTION.is_file(), f"Missing previous completed search: {PREVIOUS_SELECTION}"


In [ ]:
#@title 3. Load the pinned code and install dependencies
if not REPO.exists():
    run_live(["git", "clone", "--depth", "1", "--filter=blob:none", "--no-checkout", "--sparse", "--progress", REPO_URL, REPO], label="clone")
else:
    assert (REPO / ".git").exists(), f"{REPO} is not a Git checkout."
    origin = subprocess.check_output(["git", "-C", str(REPO), "remote", "get-url", "origin"], text=True).strip()
    assert origin == REPO_URL
    dirty = subprocess.check_output(["git", "-C", str(REPO), "status", "--porcelain", "--untracked-files=no"], text=True)
    assert not dirty, "Save local tracked-file edits before rerunning setup."
if subprocess.run(["git", "-C", str(REPO), "cat-file", "-e", REPO_COMMIT + "^{commit}"], capture_output=True).returncode:
    run_live(["git", "-C", REPO, "fetch", "origin", REPO_COMMIT], label="fetch")
run_live(["git", "-C", REPO, "sparse-checkout", "set", "experiments", "methods", "scripts"], label="checkout-files")
run_live(["git", "-C", REPO, "checkout", "--detach", REPO_COMMIT], label="checkout-revision")
# Preserve Colab's installed CUDA-compatible torch and torchvision pair.
run_live([sys.executable, "-m", "pip", "install", "-r", REPO / "requirements-modern.txt", "pandas", "matplotlib"], label="dependencies")
run_live([sys.executable, "-u", "-c", "import torch, torchvision; from experiments.multimnist.trainer import parser; print('Imports OK:', torch.__version__, torchvision.__version__)"], cwd=REPO, label="import-check")
os.chdir(REPO)
def run_script(filename, *args):
    return run_live([sys.executable, "-u", REPO / filename, *args], cwd=REPO, label=Path(filename).stem)

def atomic_json(path, value):
    tmp = path.with_name(path.name + ".tmp")
    tmp.write_text(json.dumps(value, indent=2) + "\n")
    tmp.replace(path)

identity = {"repo": REPO_URL, "commit": REPO_COMMIT, "epochs": EPOCHS,
            "batch_size": BATCH_SIZE, "seeds": SEEDS, "dataset_seed": 2026,
            "train_samples": 10000, "test_samples": 1000,
            "baseline_root": str(BASELINE_ROOT)}
manifest = RUN_ROOT / "workflow.json"
if manifest.exists():
    assert json.loads(manifest.read_text()) == identity, "Workflow settings changed. Use a new SEARCH_NAME."
else:
    atomic_json(manifest, identity)
versions = subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True)
(RUN_ROOT / ("environment_" + time.strftime("%Y%m%d_%H%M%S") + ".txt")).write_text(versions)
print("Pinned code revision:", REPO_COMMIT)
print("Existing baselines are read from:", BASELINE_ROOT)
print("New Ours outputs are written to:", RUN_ROOT)

In [ ]:
#@title 4. Restore the exact existing dataset
CACHE = DRIVE_ROOT / DATA_CACHE_EXPERIMENT / "data"
ARCHIVE = CACHE / "multimnist_seed2026.zip"
assert ARCHIVE.is_file(), f"Locate the existing dataset archive: {ARCHIVE}"
LOCAL_ARCHIVE = Path("/content/multimnist_seed2026.zip")
print("Copying the existing dataset archive from Drive...", flush=True)
shutil.copy2(ARCHIVE, LOCAL_ARCHIVE)
digest = hashlib.sha256(LOCAL_ARCHIVE.read_bytes()).hexdigest()
assert (CACHE / "dataset_archive.sha256").read_text().strip() == digest, "Archive checksum mismatch."
marker = DATA.parent / "archive.sha256"
if not (DATA.exists() and marker.exists() and marker.read_text().strip() == digest):
    if DATA.exists():
        shutil.rmtree(DATA)
    DATA.mkdir(parents=True)
    with zipfile.ZipFile(LOCAL_ARCHIVE) as archive:
        assert archive.testzip() is None
        for entry in archive.infolist():
            assert (DATA / entry.filename).resolve().is_relative_to(DATA.resolve()), "Unsafe archive entry."
        print("Extracting dataset...", flush=True)
        archive.extractall(DATA)
    marker.write_text(digest + "\n")
for split, expected in [("train", 10000), ("test", 1000)]:
    rows = list(csv.reader((DATA / split / "labels.csv").open()))
    assert len(rows) == expected
    assert all((DATA / split / "2" / row[0]).is_file() for row in rows)
    print(split, len(rows), "images", flush=True)
print("Dataset ready:", DATA)


In [ ]:
#@title 5. Verify saved baselines, data and the previous selected configuration
from IPython.display import display
import pandas as pd
import importlib
# A live Colab kernel may still cache packages from the previous checkout.
active_repo = REPO.resolve()
active_commit = subprocess.check_output(
    ["git", "-C", str(active_repo), "rev-parse", "HEAD"], text=True).strip()
if active_commit != REPO_COMMIT:
    raise RuntimeError("The active checkout is not the pinned revision. Rerun cells 1–4 of this notebook.")
sys.path[:] = [entry for entry in sys.path if entry != str(active_repo)]
sys.path.insert(0, str(active_repo))
for module_name in list(sys.modules):
    if module_name in {"experiments", "methods"} or module_name.startswith(("experiments.", "methods.")):
        del sys.modules[module_name]
importlib.invalidate_caches()
from experiments.multimnist import trainer as active_trainer
if Path(active_trainer.__file__).resolve() != active_repo / "experiments/multimnist/trainer.py":
    raise RuntimeError("Python imported a different checkout. Restart the session and rerun cells 1–5.")
training_parser = active_trainer.parser
run_signature = active_trainer.run_signature
data_fingerprint = active_trainer.data_fingerprint
print("Training module:", active_trainer.__file__)
COMMON = ["--data-path", DATA, "--device", "cuda:0", "--epochs", EPOCHS,
          "--batch-size", BATCH_SIZE, "--workers", 0, "--threads", 4, "--cache-data"]
DATA_SHA256 = data_fingerprint(DATA)
TRAIN_SHA256 = data_fingerprint(DATA, splits=("train",))

def settings_flags(settings):
    keys = ["lr", "eta", "alpha", "oracle", "weights", "momentum", "lr_schedule", "min_lr_ratio", "cooldown_start"]
    return [part for k in keys if k in settings for part in ("--"+k.replace("_", "-"), settings[k])]

def load_selected(path):
    selection = json.loads(path.read_text())
    assert selection["test_used"] is False and selection["settings"]["selection"] == "validation"
    assert selection["settings"]["train_sha256"] == TRAIN_SHA256, "Selection used different images."
    selected = selection["selected"]
    assert selected["epochs"] == EPOCHS and selected["batch_size"] == BATCH_SIZE
    return selection, selected

def training_args(root, tag, method, seed, flags):
    return ["--output-root", root, "--tag", tag, "--method", method, "--seed", seed,
            *COMMON, *flags, "--resume"]

def status_for(root, tag, method, seed, flags):
    directory = root / tag / method / f"seed{seed}"
    args = training_parser().parse_args(list(map(str, training_args(root, tag, method, seed, flags))))
    expected = run_signature(args, DATA_SHA256)
    for name in ("config.json", "summary.json"):
        path = directory / name
        if path.exists() and json.loads(path.read_text()).get("signature") != expected:
            raise ValueError(f"Configuration/data mismatch: {path}")
    result = dict(tag=tag, method=method, seed=seed, status="pending", epoch=0)
    summary_path = directory / "summary.json"
    if summary_path.exists():
        s = json.loads(summary_path.read_text())
        assert s["seed"] == seed and s["method"] == method and s["tag"] == tag
        assert s.get("selection") == "test" and not s.get("smoke")
        assert 0 <= s["epoch"] <= EPOCHS
        complete = s.get("completed") is True and s["epoch"] == EPOCHS
        result.update(status="complete" if complete else "resume", epoch=s["epoch"])
        if not complete and not (directory / "checkpoint.pt").exists():
            raise FileNotFoundError(f"Partial run has no checkpoint: {directory}")
    return result

def run_ours_jobs(root, jobs):
    for tag, flags in jobs:
        for seed in SEEDS:
            status = status_for(root, tag, "ours", seed, flags)
            print(f"{tag}/ours/seed{seed}: {status['status']}", flush=True)
            if status["status"] == "complete":
                continue
            run_script("run_multimnist.py", *training_args(root, tag, "ours", seed, flags))
            assert status_for(root, tag, "ours", seed, flags)["status"] == "complete"

BASELINE_CONFIGS = {
    "moon": ["--lr", 0.001, "--weight-lr", 0.0001, "--gamma", 0.001],
    "famo": ["--lr", 0.001, "--weight-lr", 0.025, "--gamma", 0.01],
    "famo_muon": ["--lr", 0.001, "--weight-lr", 0.025, "--gamma", 0.01],
    "mgda": ["--lr", 0.001], "mgda_muon": ["--lr", 0.001], "muon_ls": ["--lr", 0.001],
}
baseline_status = pd.DataFrame([status_for(BASELINE_ROOT, "main", method, seed, flags)
    for method, flags in BASELINE_CONFIGS.items() for seed in SEEDS])
display(baseline_status)
assert (baseline_status.status == "complete").all(), "Locate all completed baselines in EXPERIMENT_NAME; this notebook will not retrain them."
old_selection, OLD_SELECTED = load_selected(PREVIOUS_SELECTION)
assert DATA_SHA256 == "a45baadea63fc6813db6e001fc91a159bf46b8534c4bdc1e98a635b217f5f00f"
assert OLD_SELECTED["lr_schedule"] == "reference"
print("Previous selected recipe:", OLD_SELECTED)
print("Previous validation score:", old_selection["validation"]["score"])
REFINE_ARGS = ["--data-path", DATA, "--previous-search", PREVIOUS_SEARCH,
               "--output-root", TUNING_ROOT, "--device", "cuda:0", "--threads", 4]
atomic_json(RUN_ROOT / "baseline_reuse.json", {"root": str(BASELINE_ROOT),
            "data_sha256": DATA_SHA256, "methods": list(BASELINE_CONFIGS), "seeds": SEEDS,
            "training_performed": False})
print("Verified. Cell 6 performs four schedule comparisons on validation only.")


In [ ]:
#@title 6. Compare four late-decay schedules on all three validation seeds
run_script("run_multimnist_refine.py", *REFINE_ARGS)
selection, SELECTED = load_selected(SELECTION_FILE)
display(pd.DataFrame([{"tag": row["tag"], "validation_mean": row["score"],
                      "seed_std": row["score_std"]} for row in selection["confirmation"]]))
print("Frozen selected recipe:", SELECTED)
print("Recipe changed:", selection["changed"])
print("Validation improvement (percentage points):", selection["improvement_over_incumbent"])


In [ ]:
#@title 7. Train the frozen selected Ours recipe on the full training split
selection, SELECTED = load_selected(SELECTION_FILE)
frozen = {"selected": SELECTED, "selection_file": str(SELECTION_FILE),
          "validation_score": selection["validation"]["score"], "changed": selection["changed"],
          "seeds": SEEDS, "data_sha256": DATA_SHA256, "commit": REPO_COMMIT}
frozen_path = RUN_ROOT / "final_configuration.json"
if frozen_path.exists():
    assert json.loads(frozen_path.read_text()) == frozen, "Final recipe changed; use a new SEARCH_NAME."
else:
    atomic_json(frozen_path, frozen)
if selection["changed"]:
    REPORT_OURS_ROOT = OUTPUT_ROOT
    run_ours_jobs(OUTPUT_ROOT, [("main", settings_flags(SELECTED))])
else:
    REPORT_OURS_ROOT = PREVIOUS_ROOT / "results" / "multimnist"
    for seed in SEEDS:
        assert status_for(REPORT_OURS_ROOT, "main", "ours", seed, settings_flags(SELECTED))["status"] == "complete"
    print("No validated improvement: reusing the previous final Ours runs.")
print("Final Ours results:", REPORT_OURS_ROOT)


In [ ]:
#@title 8. Report the comparison and bundle histories, checkpoints and settings
REPORT = RUN_ROOT / "comparison_report"
run_script("report_multimnist.py", "--root", BASELINE_ROOT, "--ours-root", REPORT_OURS_ROOT,
           "--section", "comparison", "--baseline-source", "reproduced",
           "--seeds", *SEEDS, "--out", REPORT)
results = pd.read_csv(REPORT / "results.csv")
assert len(results) == 7 and (results.n_seeds == 3).all(), "Missing final results."
display(results[["method", "n_seeds", "left", "right", "avg", "avg_std"]])
atomic_json(REPORT / "ours_search_protocol.json", {"settings": selection["settings"],
            "selected": selection["selected"], "baseline_search": "Existing released-code settings; no additional baseline tuning"})
BUNDLE = RUN_ROOT / "multimnist_focused_comparison.zip"
print("Bundling existing and new histories/checkpoints for diagnosis...", flush=True)
paths = list(REPORT.glob("*")) + [SELECTION_FILE, TUNING_ROOT / "refinement.json", frozen_path]
for method in BASELINE_CONFIGS:
    for seed in SEEDS:
        folder = BASELINE_ROOT / "main" / method / f"seed{seed}"
        paths += [folder / name for name in ("config.json", "summary.json", "history.json", "checkpoint.pt")]
for seed in SEEDS:
    folder = REPORT_OURS_ROOT / "main" / "ours" / f"seed{seed}"
    paths += [folder / name for name in ("config.json", "summary.json", "history.json", "checkpoint.pt")]
paths += list(TUNING_ROOT.glob("*/ours/seed*/*"))
with zipfile.ZipFile(BUNDLE, "w", zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(set(paths)):
        if path.is_file():
            archive.write(path, path.relative_to(BASE_ROOT))
print("Comparison CSV:", REPORT / "results.csv")
print("Comparison LaTeX:", REPORT / "table.tex")
print("Share this complete diagnostic bundle:", BUNDLE)
